In [20]:
import os
import json
import requests
from bs4 import BeautifulSoup

In [12]:
product_code = "124893467"
page = 1
url = f"https://www.ceneo.pl/{product_code}/opinie-{page}"

In [13]:
response = requests.get(url)
print(f"Status code: {response.status_code}")

Status code: 200


In [14]:
page_dom = BeautifulSoup(response.text, "html.parser")
print(type(page_dom))

<class 'bs4.BeautifulSoup'>


In [15]:
product_name = page_dom.select_one("h1.product-top__product-info__name").get_text()
print(product_name)

Drukarka HP Color Laser 150nw (4ZB95A)


In [16]:
opinions = page_dom.find_all("div", {'class' : 'js_product-review'})
print(type(opinions))
print(len(opinions))

<class 'bs4.element.ResultSet'>
11


In [17]:
opinions = page_dom.select("div.js_product-review:not(.user-post--highlight)")
print(type(opinions))
print(len(opinions))

<class 'bs4.element.ResultSet'>
10


In [18]:
all_opinions = []

for opinion in opinions:

    def get_text(selector):
        element = opinion.select_one(selector)
        return element.text.strip() if element else None

    def get_list(selector):
        elements = opinion.select(selector)
        return [e.text.strip() for e in elements] if elements else []

    single_opinion = {
        "opinion_id": opinion.get("data-entry-id"),
        "author": get_text("span.user-post__author-name"),
        "recommendation": get_text("span.user-post__author-recomendation > em"),
        "stars": get_text("span.user-post__score-count"),
        "content": get_text("div.user-post__text"),
        "pros": get_list("div.review-feature__title--positives ~ div.review-feature__item"),
        "cons": get_list("div.review-feature__title--negatives ~ div.review-feature__item"),
        "useful": get_text("button.vote-yes > span"),
        "useless": get_text("button.vote-no > span"),
        "publish_date": get_text("span.user-post__published > time:nth-child(1)"),
        "purchase_date": get_text("span.user-post__published > time:nth-child(2)")
    }
    
    all_opinions.append(single_opinion)

print(f"Successfully extracted {len(all_opinions)} opinions.")
import json
print(json.dumps(all_opinions[0], indent=4, ensure_ascii=False))

Successfully extracted 10 opinions.
{
    "opinion_id": "13484305",
    "author": "f...s",
    "recommendation": "Polecam",
    "stars": "4,5/5",
    "content": "Produkt bardzo kompaktowy, co pozwala umieścić go nawet w niewielkich przestrzeniach pod biurkiem (należy pamiętać o wentylacji). Jakość wydruku więcej niż zadowalająca. Drukuje na papierze ozdobnym, ciężkim, na materiałach termotransferowych, etykietach i foliach. Ciężko znaleźć coś na czym nie utrwali wydruku (doskonale radzi sobie z podkładem pod etykiety do transferu na żelowe medium akrylowe (np. nadruki na drewnie bez zdrapywania/rolowania papieru)). Dużą zaletą jest też możliwość taniego nabycia modyfikowanego firmware, które umożliwia drukowanie na zasypce - wtedy strona kolor z papierem, zasypką i prądem oraz amortyzacją samej drukarki wychodzi około 1gr. Ciężko przyczepić się do jakości wydruku. Przy dobrym materiale źródłowym i papierze innym niż \"eko\" jakość jest bardziej niż zadowalająca. Ręczny dupleks również 

In [19]:
for opinion in opinions:
    single_opinion = {
        'opinion_id': opinion['data-entry-id'],
        'author': opinion.select_one('span.user-post__author-name').get_text().strip(),
        'recommendation': opinion.select_one('span.user-post__author-recomendation > em').get_text().strip()
        if opinion.select_one('span.user-post__author-recomendation > em') else None,
        'score': opinion.select_one('span.user-post__score-count').get_text().strip(),
        'content': opinion.select_one('div.user-post__text').get_text().strip(),
        'pros': [p.get_text().strip() for p in opinion.select('div.review-feature__item--positive')],
        'cons': [c.get_text().strip() for c in opinion.select('div.review-feature__item--negative')],
        'like': opinion.select_one('button.vote-yes > span').get_text().strip(),
        'dislike': opinion.select_one('button.vote-no > span').get_text().strip(),
        'publish_date': opinion.select_one('span.user-post__published > time:nth-child(1)[datetime]').get_text().strip(),
        'purchase_date': opinion.select_one('span.user-post__published > time:nth-child(2)[datetime]').get_text().strip()
        if opinion.select_one('span.user-post__published > time:nth-child(2)[datetime]') else None,
    }
    all_opinions.append(single_opinion)

{'opinion_id': '13484305', 'author': 'f...s', 'recommendation': 'Polecam', 'score': '4,5/5', 'content': 'Produkt bardzo kompaktowy, co pozwala umieścić go nawet w niewielkich przestrzeniach pod biurkiem (należy pamiętać o wentylacji). Jakość wydruku więcej niż zadowalająca. Drukuje na papierze ozdobnym, ciężkim, na materiałach termotransferowych, etykietach i foliach. Ciężko znaleźć coś na czym nie utrwali wydruku (doskonale radzi sobie z podkładem pod etykiety do transferu na żelowe medium akrylowe (np. nadruki na drewnie bez zdrapywania/rolowania papieru)). Dużą zaletą jest też możliwość taniego nabycia modyfikowanego firmware, które umożliwia drukowanie na zasypce - wtedy strona kolor z papierem, zasypką i prądem oraz amortyzacją samej drukarki wychodzi około 1gr. Ciężko przyczepić się do jakości wydruku. Przy dobrym materiale źródłowym i papierze innym niż "eko" jakość jest bardziej niż zadowalająca. Ręczny dupleks również jest kompensowany przez wymiary, a czas wydruku pierwszej s

In [ ]:
next = True if page_dom.select_one('button.pagination__next') else False
if next: page += 1

In [ ]:
if not os.path.exists("./opinions"):
    os.mkdir("./opinions")

In [ ]:
with open(f"./{product_code}.json","w",encoding = "UTF-8") as jf:
    json.dump(all_opinions,jf,indent=4,ensure_ascii=False)